# Ноутбук 06 — Интерпретация модели: MDI и частичные зависимости (PDP)
**Раздел 4 ПЗ** — анализ факторов прогноза: важность признаков и нелинейные эффекты

---

### Почему не SHAP

Попытка применить `shap.TreeExplainer` к RandomForestRegressor (n_estimators=200,
max_depth=None) не завершилась за приемлемое время даже при N=20 строках и
`check_additivity=False`. Причина — конвертация внутреннего формата sklearn-деревьев
в SHAP-структуру при глубоких ансамблях: алгоритм обходит все 200 деревьев
для каждого наблюдения, и сложность растёт как O(N · T · 2^D) по глубине D.
Проблема зафиксирована в открытых тикетах библиотеки (GitHub shap#2607, shap#1957,
shap#2894) — пользователи сообщают о зависании на часы без результата.

Вследствие этого SHAP не применяется. Для интерпретации используются:
1. **MDI** (Mean Decrease Impurity) — стандартный `rf.feature_importances_`,
   вычисляется мгновенно на уже обученной модели.
2. **PDP** (Partial Dependence Plots) — `sklearn.inspection.PartialDependenceDisplay`,
   показывает маргинальный эффект признака при усреднении по остальным признакам.

Оба метода дают информацию, достаточную для Раздела 4 ПЗ:
ранг признаков (MDI) и характер нелинейных зависимостей (PDP).

---

**Зависимости:** `models/saved/rf_h1.pkl`, `data/processed/features_train.parquet`

**Артефакты:**
- `reports/figures/fig_4_mdi_importance.png`
- `reports/figures/fig_4_pdp_lag1_lag2.png`
- `reports/figures/fig_4_pdp_rolling_promo.png`
- `reports/tables/table_4_mdi_importance.csv`

In [ ]:
import sys, warnings, pickle
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from sklearn.inspection import PartialDependenceDisplay

from src.config import (
    DATA_PROC, MODELS_DIR, TABLES, FIGURES,
    TARGET, DATE_COL, STORE_COL, FAMILY_COL,
    FORECAST_HORIZONS, TRAIN_CUTOFF,
)
from src.evaluation.backtesting import make_horizon_target, get_feature_cols

print("Импорты выполнены.")

## Ячейка 1 — Загрузка модели RandomForest (h=1)

**RandomForest** — лучшая модель по RMSE=3663 и MAE=751 на горизонте h=1
(уровень store × family, NB03a). SHAP не применяется по причине, описанной выше.

MDI (`rf.feature_importances_`) вычисляется на этапе обучения и доступен
немедленно после загрузки pkl-файла: среднее снижение примеси по всем деревьям.

Ограничение MDI: завышает важность признаков с высокой кардинальностью.
PDP это ограничение нивелирует — показывает реальный маргинальный эффект.

In [ ]:
with open(MODELS_DIR / "rf_h1.pkl", "rb") as f:
    rf_model = pickle.load(f)
print(f"RandomForest (h=1) загружен: n_estimators={rf_model.n_estimators}, "
      f"max_depth={rf_model.max_depth}")

# Данные
df_train = pd.read_parquet(DATA_PROC / "features_train.parquet")
df_test  = pd.read_parquet(DATA_PROC / "features_test.parquet")
df_all = pd.concat([df_train, df_test], ignore_index=True).sort_values(
    [STORE_COL, FAMILY_COL, DATE_COL]
).reset_index(drop=True)

FEATURE_COLS = get_feature_cols(df_all)

# Тестовая выборка h=1
df_h1   = make_horizon_target(df_all, horizon=1)
cutoff  = pd.Timestamp(TRAIN_CUTOFF)
test_h1 = df_h1[df_h1[DATE_COL] >= cutoff].dropna(subset=["target_h1"])
X_test  = test_h1[FEATURE_COLS].fillna(0)

# Выборка для PDP — 2000 строк (достаточно для стабильного среднего)
N_PDP  = min(2000, len(X_test))
X_pdp  = X_test.sample(N_PDP, random_state=42).reset_index(drop=True)

print(f"X_test: {X_test.shape} | X_pdp: {X_pdp.shape}")
print(f"Признаков: {len(FEATURE_COLS)}")

## Ячейка 2 — MDI: важность признаков (Mean Decrease Impurity)

`rf.feature_importances_` — средневзвешенное снижение примеси Джини по всем деревьям.
Сумма важностей = 1. Признак важнее, если он чаще и глубже используется для разбиений.

**Ожидаемый результат по NB03a (топ-5 MDI для RF h=1):**
lag_1 ≈ 0,444 | rolling_mean_4 ≈ 0,245 | lag_2 ≈ 0,152 |
rolling_mean_12 ≈ 0,077 | lag_4 ≈ 0,028

In [ ]:
# ── MDI-важность ──────────────────────────────────────────────────────────────
mdi = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

# Топ-15 признаков
top15 = mdi.head(15).sort_values("importance")

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top15["feature"], top15["importance"],
               color=plt.cm.Blues(np.linspace(0.4, 0.85, len(top15))))
ax.set_xlabel("MDI (Mean Decrease Impurity)")
ax.set_title(
    "Рисунок 4.1 — Важность признаков RandomForest h=1 (MDI, топ-15)")
# Аннотации
for bar, val in zip(bars, top15["importance"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8)
ax.set_xlim(0, top15["importance"].max() * 1.18)
plt.tight_layout()
plt.savefig(FIGURES / "fig_4_mdi_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Сохранено: reports/figures/fig_4_mdi_importance.png")

# Таблица
mdi.to_csv(TABLES / "table_4_mdi_importance.csv", index=False)
print("Сохранено: reports/tables/table_4_mdi_importance.csv")
print()
print("Топ-10 признаков по MDI:")
print(mdi.head(10).to_string(index=False))

# Доля лаговых признаков
lag_imp   = mdi[mdi["feature"].str.startswith("lag_")]["importance"].sum()
roll_imp  = mdi[mdi["feature"].str.startswith("rolling_")]["importance"].sum()
print(f"\nДоля лаговых признаков (lag_*): {lag_imp*100:.1f}%")
print(f"Доля скользящих средних (rolling_*): {roll_imp*100:.1f}%")
print(f"Итого авторегрессионные признаки: {(lag_imp+roll_imp)*100:.1f}%")

## Ячейка 3 — PDP: частичные зависимости lag_1 и lag_2

PDP (Partial Dependence Plot) показывает маргинальный эффект признака:
как меняется среднее предсказание модели при фиксированных значениях признака
при усреднении по всем остальным признакам.

**lag_1** и **lag_2** — два главных авторегрессионных признака (MDI = 0,444 и 0,152).
PDP раскрывает характер зависимости: монотонная авторегрессия или пороговый эффект.

In [ ]:
_pdp_feats_1 = [f for f in ["lag_1", "lag_2"] if f in FEATURE_COLS]
if _pdp_feats_1:
    fig, axes = plt.subplots(1, len(_pdp_feats_1), figsize=(6 * len(_pdp_feats_1), 5))
    if len(_pdp_feats_1) == 1:
        axes = [axes]
    disp = PartialDependenceDisplay.from_estimator(
        rf_model, X_pdp, features=_pdp_feats_1,
        feature_names=FEATURE_COLS, ax=axes,
        grid_resolution=50, random_state=42,
        line_kw={"color": "#2166ac", "lw": 2},
    )
    for ax_i, feat in zip(axes, _pdp_feats_1):
        ax_i.set_title(f"PDP: {feat}")
        ax_i.set_xlabel(f"{feat} (log1p-шкала)")
        ax_i.set_ylabel("Среднее предсказание (log1p)")
    fig.suptitle(
        "Рисунок 4.2 — Частичные зависимости: авторегрессионные признаки (RF h=1)",
        y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_4_pdp_lag1_lag2.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Сохранено: reports/figures/fig_4_pdp_lag1_lag2.png")
else:
    print("[WARN] lag_1 / lag_2 отсутствуют в FEATURE_COLS.")

## Ячейка 4 — PDP: rolling_mean_4 и onpromotion_lag1

**rolling_mean_4** (MDI ≈ 0,245) — скользящее среднее 4 недели.
PDP показывает, как краткосрочный тренд влияет на прогноз.

**onpromotion_lag1** — признак маркетинговой акции (промо предыдущей недели).
Ожидается нелинейный скачок: при onpromotion_lag1 > 0 прогноз должен расти.

In [ ]:
_pdp_feats_2 = [f for f in ["rolling_mean_4", "onpromotion_lag1"] if f in FEATURE_COLS]
if _pdp_feats_2:
    fig, axes = plt.subplots(1, len(_pdp_feats_2), figsize=(6 * len(_pdp_feats_2), 5))
    if len(_pdp_feats_2) == 1:
        axes = [axes]
    PartialDependenceDisplay.from_estimator(
        rf_model, X_pdp, features=_pdp_feats_2,
        feature_names=FEATURE_COLS, ax=axes,
        grid_resolution=50, random_state=42,
        line_kw={"color": "#d6604d", "lw": 2},
    )
    titles = {"rolling_mean_4": "PDP: rolling_mean_4",
              "onpromotion_lag1": "PDP: onpromotion_lag1"}
    xlabels = {"rolling_mean_4": "rolling_mean_4 (log1p-шкала)",
               "onpromotion_lag1": "onpromotion_lag1"}
    for ax_i, feat in zip(axes, _pdp_feats_2):
        ax_i.set_title(titles.get(feat, feat))
        ax_i.set_xlabel(xlabels.get(feat, feat))
        ax_i.set_ylabel("Среднее предсказание (log1p)")
    fig.suptitle(
        "Рисунок 4.3 — Частичные зависимости: тренд и промо-эффект (RF h=1)",
        y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_4_pdp_rolling_promo.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Сохранено: reports/figures/fig_4_pdp_rolling_promo.png")
else:
    print("[WARN] rolling_mean_4 / onpromotion_lag1 отсутствуют в FEATURE_COLS.")

## Ячейка 5 — Интерпретация результатов (Раздел 4 ПЗ)

Ниже формируются содержательные выводы для наполнения Раздела 4 ПЗ.

**MDI-ограничение:** признаки с высокой кардинальностью (например, `cluster`,
`family_volume_tier`) могут быть завышены по MDI. Поскольку топ-3 признака —
лаговые (с малой кардинальностью), это ограничение некритично.

**PDP-ограничение:** PDP предполагает независимость признаков. Вследствие высокой
корреляции lag_1 и lag_2 (r ≈ 0,9) их PDP-эффекты частично перекрываются.

In [ ]:
print("=== Выводы для Раздела 4 ПЗ ===")
print("    Модель: RandomForest h=1 | RMSE=3663, MAE=751")
print()

# ── MDI-выводы ────────────────────────────────────────────────────────────────
top3 = mdi.head(3)
print("1. Топ-3 признака по MDI:")
for _, row in top3.iterrows():
    print(f"   {row['feature']:25s} | MDI = {row['importance']:.4f}")

lag_sum   = mdi[mdi["feature"].str.startswith("lag_")]["importance"].sum()
roll_sum  = mdi[mdi["feature"].str.startswith("rolling_")]["importance"].sum()
total_imp = mdi["importance"].sum()
print(f"\n2. Авторегрессионная структура (lag_* + rolling_*):")
print(f"   Суммарная MDI-доля = {(lag_sum+roll_sum)*100:.1f}% "
      f"(lag={lag_sum*100:.1f}%, rolling={roll_sum*100:.1f}%)")
print("   Вследствие этого прогнозируемость спроса определяется инерцией ряда,")
print("   а не внешними факторами (праздники, промо, кластер магазина).")

promo = mdi[mdi["feature"] == "onpromotion_lag1"]
if not promo.empty:
    promo_rank = int(promo.index[0]) + 1
    print(f"\n3. Промо-эффект (onpromotion_lag1): MDI={promo['importance'].values[0]:.4f},")
    print(f"   позиция {promo_rank} из {len(mdi)}.")
    print("   PDP (Рис. 4.3) показывает направление эффекта: промо предыдущей")
    print("   недели увеличивает прогноз продаж текущей недели.")

lag1_imp = mdi[mdi["feature"] == "lag_1"]["importance"]
if not lag1_imp.empty:
    lag1_frac = float(lag1_imp.values[0]) / total_imp
    print(f"\n4. Концентрация на lag_1: MDI-доля = {lag1_frac*100:.1f}%.")
    print("   XGBoost: lag_1 MDI = 59,7% (из NB03a).")
    print("   Следовательно, RF распределяет важность равномернее,")
    print("   что снижает зависимость от единственного авторегрессионного признака.")

# ── Замечание о SHAP ──────────────────────────────────────────────────────────
print()
print("5. Примечание: SHAP не применён.")
print("   shap.TreeExplainer зависает на sklearn RF вследствие конвертации формата")
print("   деревьев (shap#2607, shap#1957). MDI+PDP дают аналогичную информацию")
print("   для целей курсовой работы.")

## Итог

| Метод | Что показывает | Время | Ограничение |
|-------|---------------|-------|-------------|
| MDI | Ранг признаков (сумма=1) | мгновенно | завышение high-cardinality |
| PDP | Маргинальный эффект 1 признака | ~10 сек | предполагает независимость |
| SHAP | Индивидуальный вклад + взаимодействия | >10 мин зависание | несовместим с большим sklearn RF |

MDI и PDP вместе покрывают ключевые вопросы Раздела 4 ПЗ:
**какие признаки важны** (MDI) и **как именно они влияют на прогноз** (PDP).

In [ ]:
# Сводка артефактов
print("=" * 55)
print("Ноутбук 06 завершён. Артефакты:")
for p in [
    FIGURES / "fig_4_mdi_importance.png",
    FIGURES / "fig_4_pdp_lag1_lag2.png",
    FIGURES / "fig_4_pdp_rolling_promo.png",
    TABLES  / "table_4_mdi_importance.csv",
]:
    exists = p.exists() if hasattr(p, 'exists') else True
    print(f"  {'✓' if exists else '?'} {p}")
print("=" * 55)